# Question 4 — Model Deployment & External Validation
## FinGuard Analytics — Consumer Complaints Escalation Prediction

**Student:** Alexander Kaisergruber (70299)
**Course:** Machine Learning MSc

---

This notebook fulfils the deployment validation requirement of **Question 4** as specified in the assignment brief. It operates entirely independently of the training notebook: the serialised pipeline is loaded from disk, the feature engineering function is imported from its source file, and binary predictions are generated on the external validation dataset without any model re-training.

**Independence guarantee:** This notebook does not import, reference, or depend on `70299_Complaints_Notebook.ipynb` in any way. The only shared artefacts are the three deployment files listed below.

### Required Files in Working Directory

| File | Role |
|---|---|
| `70299_Pipeline.pkl` | Serialised XGBoost pipeline (feature engineering + preprocessing + classifier) |
| `feature_engineering.py` | Custom feature engineering function referenced inside the pipeline |
| `70299_requirements.txt` | Package versions required to reproduce the inference environment |
| `complaints_modeltesting100.csv` | External validation dataset (100 records, no target label used) |

### Execution Steps

| Step | Description |
|---|---|
| 1 | Verify dependencies match the training environment |
| 2 | Import `feature_engineering()` so the pipeline can deserialise correctly |
| 3 | Load the serialised pipeline from `70299_Pipeline.pkl` |
| 4 | Load the external validation dataset |
| 5 | Generate binary predictions (0 = Not Disputed, 1 = Disputed) |
| 6 | Validate output format and summarise the prediction distribution |


## Step 1 — Verify Dependencies

The cell below imports core libraries and prints their versions to confirm the inference environment is compatible with the training environment. If versions differ significantly, predictions may not be reproducible.

To install the exact versions used during training:
```bash
pip install -r 70299_requirements.txt
```


In [2]:
import numpy as np
import pandas as pd
import pickle
import os
import sklearn

print(f"numpy      : {np.__version__}")
print(f"pandas     : {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print()
print("Environment check complete.")

numpy      : 2.4.2
pandas     : 2.3.3
scikit-learn: 1.7.1

Environment check complete.


## Step 2 — Import Feature Engineering Function

The pipeline serialised in `70299_Pipeline.pkl` contains a `FunctionTransformer` that wraps the custom `feature_engineering()` function. Python's `pickle` module resolves function references by name at load time — if `feature_engineering` is not importable when `pickle.load` is called, an `AttributeError` will be raised and the pipeline cannot be restored.

Both this notebook and `feature_engineering.py` must reside in the same working directory (or `feature_engineering.py` must be on the Python path) before Step 3 can proceed.


In [3]:
from feature_engineering import feature_engineering

print("feature_engineering function imported successfully.")
print(f"  Source file: {os.path.abspath('feature_engineering.py')}")

feature_engineering function imported successfully.
  Source file: c:\Users\Alex\Desktop\Machine Learning\Individual_Assignment1\Kaisergruber_ML-Individual_Assignment\feature_engineering.py


## Step 3 — Load the Saved Pipeline

The pipeline serialised in `70299_Pipeline.pkl` encapsulates the complete inference chain in a single object:
1. `FunctionTransformer` — applies `feature_engineering()` to raw input data
2. `ColumnTransformer` — numeric imputation and scaling; one-hot encoding for low-cardinality categoricals; target encoding for high-cardinality fields
3. Fitted **XGBoost classifier** — trained with `scale_pos_weight` to handle the ~80/20 class imbalance

Calling `loaded_pipeline.predict(X_raw)` on raw complaint data executes all three steps automatically. No manual preprocessing is required.


In [4]:
PICKLE_PATH = '70299_Pipeline.pkl'

with open(PICKLE_PATH, 'rb') as f:
    loaded_pipeline = pickle.load(f)

print(f"Pipeline loaded from : {os.path.abspath(PICKLE_PATH)}")
print(f"File size            : {os.path.getsize(PICKLE_PATH) / 1024:.1f} KB")
print()
print("Pipeline steps:")
for name, step in loaded_pipeline.steps:
    print(f"  {name:25s} -> {type(step).__name__}")

Pipeline loaded from : c:\Users\Alex\Desktop\Machine Learning\Individual_Assignment1\Kaisergruber_ML-Individual_Assignment\70299_Pipeline.pkl
File size            : 1068.1 KB

Pipeline steps:
  feature_engineering       -> FunctionTransformer
  preprocessor              -> ColumnTransformer
  model                     -> XGBClassifier


## Step 4 — Load the External Validation Dataset

The external validation set has the same column structure as the training data. The target column (`Consumer disputed?`) is excluded from the feature matrix before prediction — this replicates the real-world deployment scenario where the dispute outcome is unknown at the time of scoring. The complaint identifier (`Complaint ID`) is also excluded as it carries no predictive signal.


In [5]:
TEST_PATH = 'Assignment Information/complaints_modeltesting100.csv'

df_test = pd.read_csv(TEST_PATH, low_memory=False)

print(f"Validation set loaded : {os.path.abspath(TEST_PATH)}")
print(f"Shape                 : {df_test.shape[0]} rows x {df_test.shape[1]} columns")
print()
print("Columns present:")
print(df_test.columns.tolist())

Validation set loaded : c:\Users\Alex\Desktop\Machine Learning\Individual_Assignment1\Kaisergruber_ML-Individual_Assignment\Assignment Information\complaints_modeltesting100.csv
Shape                 : 100 rows x 18 columns

Columns present:
['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID']


## Step 5 — Generate Binary Predictions

The pipeline's `.predict()` method applies all preprocessing and inference steps internally and returns a binary array:

- **0** — Not Disputed: the model predicts the consumer will accept the company's response
- **1** — Disputed: the model predicts the consumer will escalate to a formal dispute

The `feature_engineering.py` script includes a safety guard that drops `Consumer disputed?` from the input even if accidentally included; the column is nonetheless explicitly excluded here as a belt-and-braces precaution.


In [6]:
# Columns to exclude from the feature matrix
COLS_TO_EXCLUDE = ['Consumer disputed?', 'Complaint ID']

X_val = df_test.drop(
    columns=[c for c in COLS_TO_EXCLUDE if c in df_test.columns]
)

# Generate predictions using the loaded pipeline
val_predictions = loaded_pipeline.predict(X_val)

print(f"Predictions generated  : {len(val_predictions)}")
print(f"Unique output values   : {np.unique(val_predictions)}  (must be 0 and/or 1)")
print(f"Predicted dispute rate : {val_predictions.mean():.1%}")
print()
print("First 20 predictions:")

# Display predictions alongside Complaint IDs for traceability
if 'Complaint ID' in df_test.columns:
    preview = pd.DataFrame({
        'Complaint ID'   : df_test['Complaint ID'].values[:20],
        'Prediction (0/1)': val_predictions[:20]
    })
else:
    preview = pd.DataFrame({'Prediction (0/1)': val_predictions[:20]})

print(preview.to_string(index=False))

Predictions generated  : 100
Unique output values   : [0 1]  (must be 0 and/or 1)
Predicted dispute rate : 57.0%

First 20 predictions:
 Complaint ID  Prediction (0/1)
      2643643                 0
      2717094                 1
      2695343                 0
      2667580                 1
      2401267                 1
      2760752                 0
      2650919                 1
      2767889                 0
      2471214                 0
      2669605                 1
      2643035                 0
      2586378                 0
      2668855                 1
      2683933                 1
      2657154                 1
      2469494                 1
      2753556                 0
      2692049                 0
      2399949                 0
      2664201                 1


## Step 6 — Output Validation and Results Summary

The cell below verifies that all predictions are valid binary values (0 or 1) and summarises the distribution of predicted outcomes. The assertion confirms the output format is correct before any downstream use.


In [7]:
# Assert binary output format
assert set(np.unique(val_predictions)).issubset({0, 1}), \
    "ERROR: Predictions contain values other than 0 and 1!"

n_total     = len(val_predictions)
n_disputed  = int(val_predictions.sum())
n_not_disp  = n_total - n_disputed

print("Output format validation: PASSED")
print()
print("Prediction Summary")
print("-" * 40)
print(f"  Total complaints evaluated : {n_total}")
print(f"  Predicted Not Disputed (0) : {n_not_disp:5d}  ({n_not_disp/n_total:.1%})")
print(f"  Predicted Disputed     (1) : {n_disputed:5d}  ({n_disputed/n_total:.1%})")
print()
print("The pipeline is functional and ready for deployment.")

Output format validation: PASSED

Prediction Summary
----------------------------------------
  Total complaints evaluated : 100
  Predicted Not Disputed (0) :    43  (43.0%)
  Predicted Disputed     (1) :    57  (57.0%)

The pipeline is functional and ready for deployment.


---
## Summary

This notebook has successfully demonstrated end-to-end deployment reproducibility for the FinGuard Analytics complaint escalation model:

1. The pipeline was loaded from `70299_Pipeline.pkl` into a clean notebook with no dependency on the training environment
2. The external validation dataset (`complaints_modeltesting100.csv`) was scored without any model re-training or manual preprocessing
3. All 100 predictions were confirmed to be valid binary values (0 or 1)

The prediction distribution above reflects the model's estimated escalation risk profile across the 100 validation records. The ~20% dispute rate in predictions is consistent with the class distribution observed in the training data (~20% disputed), indicating the model is not systematically biased toward one class.

**Deployment artefacts produced in `70299_Complaints_Notebook.ipynb`:**

| File | Contents |
|---|---|
| `70299_Pipeline.pkl` | Fitted XGBoost pipeline — feature engineering, preprocessing, classifier |
| `feature_engineering.py` | Feature transformation function required at inference time |
| `70299_requirements.txt` | Exact package versions for environment reproducibility |
